In [3]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import re
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# =========================================================
# 0. パス設定（整理版）
# =========================================================
CONFIG_PATH = Path("config.yaml")

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    BASE = Path(config["data_root"]).expanduser()
else:
    BASE = Path("~/Desktop/Research/Data").expanduser()
    
RAW_CENSUS = BASE / "Raw" / "Census2020"
SMALLAREA_GPKG = BASE / "Processed" / "Geospatial" / "Town_polygon" / "2020" / "smallarea_2020_alljapan.gpkg"

TABLE6_DIR = RAW_CENSUS / "table6_1_household"
TABLE7_DIR = RAW_CENSUS / "table7_1_housing"
TABLE9_DIR = RAW_CENSUS / "table9_labor_force"
TABLE12_DIR = RAW_CENSUS / "table12_occupation"

# 👇 smallarea専用フォルダに変更
OUT_DIR = BASE / "Processed" / "Census2020" / "smallarea"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECK_DIR = BASE / "Intermediate" / "Census2020" / "alljapan_check_final_unit"
CHECK_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 出力ファイル（命名整理）
# =========================================================

# 属性テーブル（geometryなし）
OUT_TABLE = OUT_DIR / "smallarea_adi_attribute_final_unit.parquet"

# full版（geometryあり）
OUT_GPKG = OUT_DIR / "smallarea_adi_full.gpkg"
OUT_PARQUET = OUT_DIR / "smallarea_adi_full.parquet"

# 軽量版
OUT_MIN_GPKG = OUT_DIR / "smallarea_adi_full_min.gpkg"
OUT_MIN_PARQUET = OUT_DIR / "smallarea_adi_full_min.parquet"

# マスターマッピング
OUT_MASTER_MAP = CHECK_DIR / "alljapan_final_unit_master_map.csv"

In [2]:
# =========================================================
# 1. 共通関数
# =========================================================
def read_census_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        dtype=str,
        encoding="cp932",
        header=3
    )
    df.columns = [str(c).strip().replace("\n", "") for c in df.columns]
    return df


def clean_symbols(df: pd.DataFrame) -> pd.DataFrame:
    """
    国勢調査の記号処理
    - '-' は後で 0 として使うことが多いので一旦そのまま
    - 'X' は秘匿として欠損
    """
    df = df.copy()
    df = df.replace({
        "－": "-",
        "—": "-",
        "―": "-",
        "X": pd.NA,
        "x": pd.NA,
        "…": pd.NA,
        "… ": pd.NA,
    })
    return df


def clean_code_string(code):
    """
    コード文字列を整形。
    先頭ゼロは保持する。
    """
    if pd.isna(code):
        return pd.NA

    s = str(code).strip()

    if s in ["", "nan", "None", "<NA>", "-"]:
        return pd.NA

    s = s.replace(".0", "")
    s = re.sub(r"[^\d]", "", s)

    if s == "":
        return pd.NA

    return s


def normalize_town_code_for_shp(code, level=None):
    """
    国勢調査の町丁字コードを、shapefile側の S_AREA/KEY_CODE に合わせて正規化する。

    基本方針:
    - 6桁以上なら先頭6桁を使う
    - 6桁未満なら、右側を0埋めして6桁にする
      （先頭ゼロは元の文字列のまま保持）
    """
    s = clean_code_string(code)
    if pd.isna(s):
        return pd.NA

    if len(s) >= 6:
        return s[:6]

    return s.ljust(6, "0")


def normalize_code_list_for_shp(code_str):
    """
    合算地域列を list[str] にして、各コードを shapefile用に正規化する。
    """
    if pd.isna(code_str):
        return []

    s = str(code_str).strip()
    if s in ["", "nan", "None", "<NA>", "-"]:
        return []

    out = []
    for x in s.split(";"):
        nx = normalize_town_code_for_shp(x)
        if pd.notna(nx):
            out.append(nx)
    return out

def prepare_common(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = clean_symbols(df)

    df["市区町村コード"] = (
        df["市区町村コード"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

    df["町丁字コード_str"] = (
        df["町丁字コード"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    df["地域階層レベル"] = (
        df["地域階層レベル"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    # ★ ここを修正
    df["町丁字コード_norm"] = [
        normalize_town_code_for_shp(c, l)
        for c, l in zip(df["町丁字コード"], df["地域階層レベル"])
    ]

    df["秘匿先情報_norm"] = df["秘匿先情報"].apply(normalize_town_code_for_shp)
    df["合算地域_list"] = df["合算地域"].apply(normalize_code_list_for_shp)

    return df

def make_node(muni_code, town_code_norm):
    if pd.isna(muni_code) or pd.isna(town_code_norm):
        return pd.NA
    return f"{muni_code}_{town_code_norm}"


def level_priority(level):
    """
    粗いレベルを優先: 2 > 3 > 4
    小さい値ほど優先
    """
    order = {"2": 0, "3": 1, "4": 2}
    return order.get(str(level), 99)

In [3]:
# =========================================================
# 2. final_unit_code を付与
# =========================================================
def attach_final_unit_code(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    parent = {}

    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        if pd.isna(x) or pd.isna(y):
            return
        rx = find(x)
        ry = find(y)
        if rx != ry:
            parent[ry] = rx

    node_info = {}

    for _, row in work.iterrows():
        muni = row["市区町村コード"]
        code = row["町丁字コード_norm"]
        node = make_node(muni, code)
        if pd.isna(node):
            continue

        node_info[node] = {
            "市区町村コード": muni,
            "町丁字コード_norm": code,
            "地域階層レベル": str(row["地域階層レベル"]).strip() if pd.notna(row["地域階層レベル"]) else None,
            "秘匿処理": str(row["秘匿処理"]).strip() if pd.notna(row["秘匿処理"]) else None,
        }

    # 秘匿地域 / 合算地域あり の関係を union
    for _, row in work.iterrows():
        muni = row["市区町村コード"]
        code = row["町丁字コード_norm"]
        status = str(row["秘匿処理"]).strip() if pd.notna(row["秘匿処理"]) else ""
        target = row["秘匿先情報_norm"]
        merged_list = row["合算地域_list"]

        node = make_node(muni, code)
        if pd.isna(node):
            continue

        find(node)

        # 秘匿地域: 自分と秘匿先を結ぶ
        if status == "秘匿地域" and pd.notna(target):
            union(node, make_node(muni, target))

        # 合算地域あり: 自分と合算地域列の各コードを結ぶ
        if status == "合算地域あり":
            for c in merged_list:
                union(node, make_node(muni, c))

    # グループ化
    all_nodes = [
        make_node(m, c)
        for m, c in zip(work["市区町村コード"], work["町丁字コード_norm"])
    ]
    all_nodes = [x for x in all_nodes if pd.notna(x)]

    groups = {}
    for node in all_nodes:
        root = find(node)
        groups.setdefault(root, []).append(node)

    # final_unit_code は「合算地域あり」行を優先
    root_to_final = {}

    for root, members in groups.items():
        infos = [node_info[m] for m in members if m in node_info]

        agg_infos = [x for x in infos if x["秘匿処理"] == "合算地域あり"]

        if len(agg_infos) == 1:
            final_code = agg_infos[0]["町丁字コード_norm"]
        elif len(agg_infos) >= 2:
            agg_infos = sorted(
                agg_infos,
                key=lambda x: (level_priority(x["地域階層レベル"]), x["町丁字コード_norm"])
            )
            final_code = agg_infos[0]["町丁字コード_norm"]
        else:
            town_codes = sorted([x["町丁字コード_norm"] for x in infos if pd.notna(x["町丁字コード_norm"])])
            final_code = town_codes[0]

        root_to_final[root] = final_code

    def assign_final_unit_code(muni, town_code_norm):
        node = make_node(muni, town_code_norm)
        if pd.isna(node):
            return pd.NA
        return root_to_final[find(node)]

    work["final_unit_code"] = [
        assign_final_unit_code(m, c)
        for m, c in zip(work["市区町村コード"], work["町丁字コード_norm"])
    ]
    work["final_key"] = work["市区町村コード"] + "_" + work["final_unit_code"]

    return work


# =========================================================
# 3. keep_finest を付与
# =========================================================
def attach_keep_finest(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    def has_finer_descendant(row, code_list):
        code = row["町丁字コード_str"]
        level = str(row["地域階層レベル"]).strip() if pd.notna(row["地域階層レベル"]) else None

        if pd.isna(code) or code in ["", "-"]:
            return False

        # lv4 は最細
        if level == "4":
            return False

        for other in code_list:
            if other == code:
                continue
            if len(str(other)) > len(str(code)) and str(other).startswith(str(code)):
                return True

        return False

    work["has_finer_descendant"] = False

    for muni, g in work.groupby("市区町村コード", sort=False):
        codes_in_muni = g["町丁字コード_str"].dropna().tolist()
        idx = g.index

        work.loc[idx, "has_finer_descendant"] = [
            has_finer_descendant(row, codes_in_muni)
            for _, row in g.iterrows()
        ]

    work["keep_finest"] = ~work["has_finer_descendant"]

    # 合算地域あり は公開単位なので残す
    mask_agg = work["秘匿処理"].astype(str).str.strip() == "合算地域あり"
    work.loc[mask_agg, "keep_finest"] = True

    return work

In [4]:
# =========================================================
# 4. ファイル探索（命名規則対応版）
# =========================================================
def find_pref_files(directory: Path, table_prefix: str, pref_code: int):
    """
    ファイル命名規則に応じて都道府県別CSVを探す

    table6_1_household: h06_01_03.csv のような形式
    table7_1_housing  : h07_01_03.csv のような形式
    table9_labor_force: h09_03.csv    のような形式
    table12_occupation: h12_03.csv    のような形式
    """
    pref = f"{pref_code:02d}"

    # table6, table7 は h06_01_03.csv / h07_01_03.csv 型
    if table_prefix in ["h06", "h07"]:
        files = sorted(directory.glob(f"{table_prefix}_*_{pref}.csv"))

    # table9, table12 は h09_03.csv / h12_03.csv 型
    elif table_prefix in ["h09", "h12"]:
        files = sorted(directory.glob(f"{table_prefix}_{pref}.csv"))

    else:
        raise ValueError(f"Unsupported table_prefix: {table_prefix}")

    return files


def read_and_concat_pref_files(directory: Path, table_prefix: str, pref_code: int) -> pd.DataFrame:
    files = find_pref_files(directory, table_prefix, pref_code)

    if len(files) == 0:
        raise FileNotFoundError(
            f"No files found for {table_prefix} pref={pref_code:02d} in {directory}"
        )

    dfs = []
    for f in files:
        d = read_census_csv(f)
        d["source_file"] = f.name
        dfs.append(d)

    out = pd.concat(dfs, ignore_index=True)
    return out

In [5]:
# =========================================================
# 5. 表ごとの前処理
# =========================================================
def prepare_table6_pref(pref_code: int) -> pd.DataFrame:
    df = read_and_concat_pref_files(TABLE6_DIR, "h06", pref_code)
    df = prepare_common(df)

    # 年齢総数のみ
    age_col = "世帯員の年齢による世帯の種類"
    if age_col in df.columns:
        df = df[df[age_col] == "総数"].copy()

    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    num_cols = [
        "総数",
        "親族のみの世帯",
        "核家族世帯",
        "うち夫婦のみの世帯",
        "うち夫婦と子供から成る世帯",
        "核家族以外の世帯",
        "非親族を含む世帯",
        "単独世帯",
        "世帯の家族類型「不詳」",
        "（再掲）3世代世帯",
        "（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯",
        "（再掲）65歳以上の単独世帯",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df


def prepare_table7_pref(pref_code: int) -> pd.DataFrame:
    df = read_and_concat_pref_files(TABLE7_DIR, "h07", pref_code)
    df = prepare_common(df)
    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    num_cols = [
        "住宅に住む一般世帯",
        "持ち家",
        "公営・都市再生機構・公社の借家",
        "民営の借家",
        "給与住宅",
        "間借り",
        "住宅以外に住む一般世帯",
        "住居の種類「不詳」",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df


def prepare_table9_pref(pref_code: int) -> pd.DataFrame:
    df = read_and_concat_pref_files(TABLE9_DIR, "h09", pref_code)
    df = prepare_common(df)

    if "男女" in df.columns:
        df = df[df["男女"] == "総数"].copy()

    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    num_cols = [
        "総数",
        "労働力人口",
        "非労働力人口",
        "労働力状態「不詳」",
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df


def prepare_table12_pref(pref_code: int) -> pd.DataFrame:
    df = read_and_concat_pref_files(TABLE12_DIR, "h12", pref_code)
    df = prepare_common(df)

    if "男女" in df.columns:
        df = df[df["男女"] == "総数"].copy()

    df = attach_final_unit_code(df)
    df = attach_keep_finest(df)
    df = df[df["keep_finest"]].copy()

    occ_cols = [
        "0_総数",
        "A_管理的職業従事者",
        "B_専門的・技術的職業従事者",
        "C_事務従事者",
        "D_販売従事者",
        "E_サービス職業従事者",
        "F_保安職業従事者",
        "G_農林漁業従事者",
        "H_生産工程従事者",
        "I_輸送・機械運転従事者",
        "J_建設・採掘従事者",
        "K_運搬・清掃・包装等従事者",
        "L_分類不能の職業",
    ]
    for c in occ_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].replace("-", 0), errors="coerce")

    return df

In [6]:
# =========================================================
# 6. final_key 単位で集約
# =========================================================
def aggregate_by_final_key(df: pd.DataFrame, sum_cols: list, table_name: str) -> pd.DataFrame:
    group_cols = ["final_key", "市区町村コード", "final_unit_code"]

    keep_meta = [
        "都道府県名", "市区町村名", "大字・町名", "字・丁目名",
        "地域階層レベル", "秘匿処理", "町丁字コード", "町丁字コード_norm"
    ]
    keep_meta = [c for c in keep_meta if c in df.columns]

    agg_dict = {c: "sum" for c in sum_cols if c in df.columns}
    for c in keep_meta:
        agg_dict[c] = "first"

    out = df.groupby(group_cols, dropna=False, as_index=False).agg(agg_dict)
    out["source_table"] = table_name
    return out

In [7]:
# =========================================================
# 7. 全国処理
# =========================================================
all_df6 = []
all_df7 = []
all_df9 = []
all_df12 = []

master_maps = []

for pref in range(1, 48):
    print(f"processing pref {pref:02d} ...")

    df6 = prepare_table6_pref(pref)
    df7 = prepare_table7_pref(pref)
    df9 = prepare_table9_pref(pref)
    df12 = prepare_table12_pref(pref)

    # 確認用に mapping 候補を貯める（keep_finest 前後どちらでもよいが、
    # polygons用には元コード→final_unit_code が必要なので prepared 後を使う）
    for d, name in [(df6, "table6"), (df7, "table7"), (df9, "table9"), (df12, "table12")]:
        mm = d[[
            "市区町村コード", "町丁字コード_norm", "final_unit_code", "final_key"
        ]].drop_duplicates().copy()
        mm["source_table"] = name
        master_maps.append(mm)

    df6a = aggregate_by_final_key(df6, [
        "総数",
        "親族のみの世帯",
        "核家族世帯",
        "うち夫婦のみの世帯",
        "うち夫婦と子供から成る世帯",
        "核家族以外の世帯",
        "非親族を含む世帯",
        "単独世帯",
        "世帯の家族類型「不詳」",
        "（再掲）3世代世帯",
        "（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯",
        "（再掲）65歳以上の単独世帯",
    ], "table6")

    df7a = aggregate_by_final_key(df7, [
        "住宅に住む一般世帯",
        "持ち家",
        "公営・都市再生機構・公社の借家",
        "民営の借家",
        "給与住宅",
        "間借り",
        "住宅以外に住む一般世帯",
        "住居の種類「不詳」",
    ], "table7")

    df9a = aggregate_by_final_key(df9, [
        "総数",
        "労働力人口",
        "非労働力人口",
        "労働力状態「不詳」",
    ], "table9").rename(columns={"総数": "pop15_total"})

    df12a = aggregate_by_final_key(df12, [
        "0_総数",
        "A_管理的職業従事者",
        "B_専門的・技術的職業従事者",
        "C_事務従事者",
        "D_販売従事者",
        "E_サービス職業従事者",
        "F_保安職業従事者",
        "G_農林漁業従事者",
        "H_生産工程従事者",
        "I_輸送・機械運転従事者",
        "J_建設・採掘従事者",
        "K_運搬・清掃・包装等従事者",
        "L_分類不能の職業",
    ], "table12")

    all_df6.append(df6a)
    all_df7.append(df7a)
    all_df9.append(df9a)
    all_df12.append(df12a)

df6_all = pd.concat(all_df6, ignore_index=True)
df7_all = pd.concat(all_df7, ignore_index=True)
df9_all = pd.concat(all_df9, ignore_index=True)
df12_all = pd.concat(all_df12, ignore_index=True)

print("df6_all:", df6_all.shape)
print("df7_all:", df7_all.shape)
print("df9_all:", df9_all.shape)
print("df12_all:", df12_all.shape)

master_map = pd.concat(master_maps, ignore_index=True).drop_duplicates()
print("master_map:", master_map.shape)



processing pref 01 ...
processing pref 02 ...
processing pref 03 ...
processing pref 04 ...
processing pref 05 ...
processing pref 06 ...
processing pref 07 ...
processing pref 08 ...
processing pref 09 ...
processing pref 10 ...
processing pref 11 ...
processing pref 12 ...
processing pref 13 ...
processing pref 14 ...
processing pref 15 ...
processing pref 16 ...
processing pref 17 ...
processing pref 18 ...
processing pref 19 ...
processing pref 20 ...
processing pref 21 ...
processing pref 22 ...
processing pref 23 ...
processing pref 24 ...
processing pref 25 ...
processing pref 26 ...
processing pref 27 ...
processing pref 28 ...
processing pref 29 ...
processing pref 30 ...
processing pref 31 ...
processing pref 32 ...
processing pref 33 ...
processing pref 34 ...
processing pref 35 ...
processing pref 36 ...
processing pref 37 ...
processing pref 38 ...
processing pref 39 ...
processing pref 40 ...
processing pref 41 ...
processing pref 42 ...
processing pref 43 ...
processing 

In [8]:
# =========================================================
# 8. 共通 final_key に絞って結合
# =========================================================
keys6 = set(df6_all["final_key"])
keys7 = set(df7_all["final_key"])
keys9 = set(df9_all["final_key"])
keys12 = set(df12_all["final_key"])

common_keys = keys6 & keys7 & keys9 & keys12
print("common_keys:", len(common_keys))

df6m = df6_all[df6_all["final_key"].isin(common_keys)].copy()
df7m = df7_all[df7_all["final_key"].isin(common_keys)].copy()
df9m = df9_all[df9_all["final_key"].isin(common_keys)].copy()
df12m = df12_all[df12_all["final_key"].isin(common_keys)].copy()

merged = df6m.merge(
    df7m[["final_key", "住宅に住む一般世帯", "持ち家", "公営・都市再生機構・公社の借家", "民営の借家", "給与住宅", "間借り", "住宅以外に住む一般世帯", "住居の種類「不詳」"]],
    on="final_key", how="left", validate="1:1"
)

merged = merged.merge(
    df9m[["final_key", "pop15_total", "労働力人口", "非労働力人口", "労働力状態「不詳」"]],
    on="final_key", how="left", validate="1:1"
)

merged = merged.merge(
    df12m[[
        "final_key", "0_総数",
        "A_管理的職業従事者", "B_専門的・技術的職業従事者", "C_事務従事者",
        "D_販売従事者", "E_サービス職業従事者", "F_保安職業従事者",
        "G_農林漁業従事者", "H_生産工程従事者", "I_輸送・機械運転従事者",
        "J_建設・採掘従事者", "K_運搬・清掃・包装等従事者", "L_分類不能の職業"
    ]],
    on="final_key", how="left", validate="1:1"
)

print("merged:", merged.shape)

# =========================================================
# 9. ADI構成変数
# =========================================================
merged["old_couple_rate"] = (
    merged["（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯"] / merged["総数"]
)

merged["old_single_rate"] = (
    merged["（再掲）65歳以上の単独世帯"] / merged["総数"]
)

merged["single_parent_proxy_rate"] = (
    (merged["核家族世帯"]
     - merged["うち夫婦のみの世帯"]
     - merged["うち夫婦と子供から成る世帯"])
    / merged["総数"]
)

merged["rent_rate"] = (
    (merged["住宅に住む一般世帯"] - merged["持ち家"])
    / merged["住宅に住む一般世帯"]
)

merged["sales_service_rate"] = (
    (merged["D_販売従事者"]
     + merged["E_サービス職業従事者"]
     + merged["F_保安職業従事者"])
    / merged["0_総数"]
)

merged["agriculture_rate"] = (
    merged["G_農林漁業従事者"] / merged["0_総数"]
)

merged["blue_collar_rate"] = (
    (merged["H_生産工程従事者"]
     + merged["I_輸送・機械運転従事者"]
     + merged["J_建設・採掘従事者"]
     + merged["K_運搬・清掃・包装等従事者"])
    / merged["0_総数"]
)

merged["unemployment_rate"] = (
    (merged["労働力人口"] - merged["0_総数"]) / merged["労働力人口"]
)

# 非居住地域フラグ
merged["exclude_nonresidential"] = (
    (merged["総数"].fillna(0) == 0) |
    (merged["住宅に住む一般世帯"].fillna(0) == 0)
)

# =========================================================
# 10. ADI計算
# =========================================================
merged["ADI_raw"] = (
    2.99 * merged["old_couple_rate"]
    + 7.57 * merged["old_single_rate"]
    + 17.4 * merged["single_parent_proxy_rate"]
    + 2.22 * merged["rent_rate"]
    + 4.03 * merged["sales_service_rate"]
    + 6.05 * merged["agriculture_rate"]
    + 5.38 * merged["blue_collar_rate"]
    + 18.3 * merged["unemployment_rate"]
)

print("ADI missing:", merged["ADI_raw"].isna().sum())
print("ADI matched:", merged["ADI_raw"].notna().sum())

merged.to_parquet(OUT_TABLE)
print("saved:", OUT_TABLE)

common_keys: 214093
merged: (214093, 49)
ADI missing: 10657
ADI matched: 203436
saved: /Users/Tomo/Desktop/Research/Data/Processed/Census2020/smallarea/smallarea_adi_attribute_final_unit.parquet


In [9]:
def normalize_key_code_for_polygon(key_code):
    """
    ポリゴン側の KEY_CODE を 11 桁に正規化する。
    - 数字以外を除去
    - 11桁未満なら右側を 0 埋め
    - 11桁超なら先頭11桁を採用
    """
    if pd.isna(key_code):
        return pd.NA

    s = str(key_code).strip()
    s = s.replace(".0", "")
    s = "".join([c for c in s if c.isdigit()])

    if s == "":
        return pd.NA

    if len(s) < 11:
        s = s.ljust(11, "0")
    elif len(s) > 11:
        s = s[:11]

    return s

# =========================================================
# 11. ポリゴン読込
# =========================================================
smallarea_gdf = gpd.read_file(SMALLAREA_GPKG)
print("smallarea_gdf:", smallarea_gdf.shape)
print("CRS:", smallarea_gdf.crs)
print(smallarea_gdf.columns.tolist())

smallarea_gdf = smallarea_gdf.copy()

# -----------------------------------
# KEY_CODE を正規化（★ここが新規）
# -----------------------------------
smallarea_gdf["KEY_CODE_norm"] = smallarea_gdf["KEY_CODE"].apply(normalize_key_code_for_polygon)

# -----------------------------------
# 市区町村コード + 町丁字コードを KEY_CODE から作る
# -----------------------------------
smallarea_gdf["市区町村コード"] = smallarea_gdf["KEY_CODE_norm"].str[:5]
smallarea_gdf["町丁字コード_norm"] = smallarea_gdf["KEY_CODE_norm"].str[5:]

# -----------------------------------
# 参考用（デバッグ用に残す）
# -----------------------------------
smallarea_gdf["PREF"] = (
    smallarea_gdf["PREF"]
    .astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(2)
)
smallarea_gdf["CITY"] = (
    smallarea_gdf["CITY"]
    .astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(3)
)
smallarea_gdf["S_AREA"] = (
    smallarea_gdf["S_AREA"]
    .astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
)

# -----------------------------------
# master_map と結合
# -----------------------------------
poly_map = (
    master_map[["市区町村コード", "町丁字コード_norm", "final_unit_code", "final_key"]]
    .drop_duplicates()
    .copy()
)

smallarea_map_gdf = smallarea_gdf.merge(
    poly_map,
    on=["市区町村コード", "町丁字コード_norm"],
    how="left"
)

# -----------------------------------
# fallback（マッチしない場合）
# -----------------------------------
smallarea_map_gdf["final_unit_code"] = smallarea_map_gdf["final_unit_code"].fillna(
    smallarea_map_gdf["町丁字コード_norm"]
)

smallarea_map_gdf["final_key"] = smallarea_map_gdf["final_key"].fillna(
    smallarea_map_gdf["市区町村コード"] + "_" + smallarea_map_gdf["final_unit_code"]
)

print("mapped polygons:", smallarea_map_gdf.shape)
print("missing final_key:", smallarea_map_gdf["final_key"].isna().sum())

smallarea_gdf: (232019, 32)
CRS: EPSG:6668
['KEY_CODE', 'PREF', 'CITY', 'S_AREA', 'PREF_NAME', 'CITY_NAME', 'S_NAME', 'KIGO_E', 'HCODE', 'AREA', 'PERIMETER', 'R2KAxx', 'R2KAxx_ID', 'KIHON1', 'DUMMY1', 'KIHON2', 'KEYCODE1', 'KEYCODE2', 'AREA_MAX_F', 'KIGO_D', 'N_KEN', 'N_CITY', 'KIGO_I', 'KBSUM', 'JINKO', 'SETAI', 'X_CODE', 'Y_CODE', 'KCODE1', 'pref_dir', 'source_shp', 'geometry']
mapped polygons: (232019, 37)
missing final_key: 0


In [10]:
# =========================================================
# 12. final_key 単位で dissolve
# =========================================================
poly_cols = [c for c in ["final_key", "市区町村コード", "final_unit_code", "PREF_NAME", "CITY_NAME"] if c in smallarea_map_gdf.columns]

smallarea_final_poly = smallarea_map_gdf[poly_cols + ["geometry"]].copy()

smallarea_final_poly = smallarea_final_poly.dissolve(
    by="final_key",
    as_index=False,
    aggfunc="first"
)

print("smallarea_final_poly:", smallarea_final_poly.shape)

# =========================================================
# 13. ADI表を結合
# =========================================================
adi_keep_cols = [
    "final_key",
    "市区町村コード",
    "都道府県名",
    "市区町村名",
    "大字・町名",
    "字・丁目名",
    "総数",
    "住宅に住む一般世帯",
    "労働力人口",
    "0_総数",
    "old_couple_rate",
    "old_single_rate",
    "single_parent_proxy_rate",
    "rent_rate",
    "sales_service_rate",
    "agriculture_rate",
    "blue_collar_rate",
    "unemployment_rate",
    "ADI_raw",
    "exclude_nonresidential",
]
adi_keep_cols = [c for c in adi_keep_cols if c in merged.columns]

smallarea_adi_gdf = smallarea_final_poly.merge(
    merged[adi_keep_cols],
    on=["final_key", "市区町村コード"],
    how="left",
    validate="1:1"
)

print("smallarea_adi_gdf:", smallarea_adi_gdf.shape)
print("missing ADI_raw:", smallarea_adi_gdf["ADI_raw"].isna().sum())

# =========================================================
# 14. 保存
# =========================================================
smallarea_adi_gdf.to_file(OUT_GPKG, driver="GPKG")
smallarea_adi_gdf.to_parquet(OUT_PARQUET)

keep_min = [c for c in ["final_key", "市区町村コード", "final_unit_code", "ADI_raw", "exclude_nonresidential", "geometry"] if c in smallarea_adi_gdf.columns]
smallarea_adi_min = smallarea_adi_gdf[keep_min].copy()

smallarea_adi_min.to_file(OUT_MIN_GPKG, driver="GPKG")
smallarea_adi_min.to_parquet(OUT_MIN_PARQUET)

print("saved:")
print(OUT_GPKG)
print(OUT_PARQUET)
print(OUT_MIN_GPKG)
print(OUT_MIN_PARQUET)

smallarea_final_poly: (214115, 6)
smallarea_adi_gdf: (214115, 24)
missing ADI_raw: 10679
saved:
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/smallarea/smallarea_adi_full.gpkg
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/smallarea/smallarea_adi_full.parquet
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/smallarea/smallarea_adi_full_min.gpkg
/Users/Tomo/Desktop/Research/Data/Processed/Census2020/smallarea/smallarea_adi_full_min.parquet


In [11]:
# =========================================================
# 15. 確認
# =========================================================
print(smallarea_adi_gdf[[
    c for c in [
        "final_key", "市区町村名", "大字・町名", "字・丁目名",
        "総数", "住宅に住む一般世帯", "労働力人口", "0_総数",
        "ADI_raw", "exclude_nonresidential"
    ] if c in smallarea_adi_gdf.columns
]].head(20))

print("\nADI summary:")
print(smallarea_adi_gdf["ADI_raw"].describe())

print("\nnonresidential flag counts:")
print(smallarea_adi_gdf["exclude_nonresidential"].value_counts(dropna=False))

       final_key   市区町村名     大字・町名 字・丁目名     総数  住宅に住む一般世帯  労働力人口   0_総数   ADI_raw exclude_nonresidential
0   01101_030000  札幌市中央区        円山  None    0.0        0.0    0.0    0.0       NaN                   True
1   01101_040000  札幌市中央区  円山西町（番地）  None    0.0        0.0    0.0    0.0       NaN                   True
2   01101_100000  札幌市中央区   宮ケ丘（番地）  None   17.0       15.0   25.0   24.0  4.987711                  False
3   01101_110000  札幌市中央区        盤渓  None   50.0       49.0   50.0   47.0  8.405695                  False
4   01101_120101  札幌市中央区     宮の森一条   一丁目   34.0       34.0   40.0   39.0  4.871135                  False
5   01101_120102  札幌市中央区     宮の森一条   二丁目   34.0       34.0   40.0   39.0  3.989287                  False
6   01101_120103  札幌市中央区     宮の森一条   三丁目  104.0      103.0   92.0   88.0  5.439261                  False
7   01101_120104  札幌市中央区     宮の森一条   四丁目  119.0      119.0   79.0   73.0  6.746092                  False
8   01101_120105  札幌市中央区     宮の森一条   五丁目   91.

In [12]:
analysis_df = smallarea_adi_gdf[
    (smallarea_adi_gdf["exclude_nonresidential"] == False) &
    (smallarea_adi_gdf["ADI_raw"].notna())
].copy()

out_path = "/Users/Tomo/Desktop/Research/Data/Processed/Census2020/smallarea/smallarea_adi_analysis.parquet"
analysis_df.to_parquet(out_path)

# parquet（分析用）
analysis_df.to_parquet(out_path)

# gpkg（GIS用）
analysis_gdf = gpd.GeoDataFrame(
    analysis_df,
    geometry="geometry",
    crs=smallarea_adi_gdf.crs
)

analysis_gdf.to_file(
    "/Users/Tomo/Desktop/Research/Data/Processed/Census2020/smallarea/smallarea_adi_analysis.gpkg",
    driver="GPKG"
)

In [13]:
nan_df = smallarea_adi_gdf[
    smallarea_adi_gdf["exclude_nonresidential"].isna()
].copy()

out_csv = "/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/debug_exclude_nan.csv"
nan_df.to_csv(out_csv, index=False)

print("saved:", out_csv)
print("rows:", len(nan_df))

saved: /Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/debug_exclude_nan.csv
rows: 22


#### debugチェック用

In [19]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

BASE = Path("/Users/Tomo/Desktop/Research/Data")
OUT_DIR = BASE / "Intermediate" / "Census2020" / "tokyo_polygon_debug"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 東京都だけ抜き出し
# ---------------------------------------------------------
tokyo_poly_debug = smallarea_map_gdf[
    smallarea_map_gdf["PREF_NAME"] == "東京都"
].copy()

print("tokyo_poly_debug:", tokyo_poly_debug.shape)
print(tokyo_poly_debug.columns.tolist())

# ---------------------------------------------------------
# geometryなしCSV（Excel確認用）
# ---------------------------------------------------------
tokyo_poly_debug_csv = tokyo_poly_debug.drop(columns="geometry", errors="ignore").copy()

out_csv = OUT_DIR / "tokyo_smallarea_map_debug.csv"
tokyo_poly_debug_csv.to_csv(out_csv, index=False, encoding="utf-8-sig")

# ---------------------------------------------------------
# geometryありGPKG（GIS確認用）
# ---------------------------------------------------------
out_gpkg = OUT_DIR / "tokyo_smallarea_map_debug.gpkg"
tokyo_poly_debug.to_file(out_gpkg, driver="GPKG")

print("saved:")
print(out_csv)
print(out_gpkg)

tokyo_poly_debug: (6021, 37)
['KEY_CODE', 'PREF', 'CITY', 'S_AREA', 'PREF_NAME', 'CITY_NAME', 'S_NAME', 'KIGO_E', 'HCODE', 'AREA', 'PERIMETER', 'R2KAxx', 'R2KAxx_ID', 'KIHON1', 'DUMMY1', 'KIHON2', 'KEYCODE1', 'KEYCODE2', 'AREA_MAX_F', 'KIGO_D', 'N_KEN', 'N_CITY', 'KIGO_I', 'KBSUM', 'JINKO', 'SETAI', 'X_CODE', 'Y_CODE', 'KCODE1', 'pref_dir', 'source_shp', 'geometry', 'KEY_CODE_norm', '市区町村コード', '町丁字コード_norm', 'final_unit_code', 'final_key']
saved:
/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_polygon_debug/tokyo_smallarea_map_debug.csv
/Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_polygon_debug/tokyo_smallarea_map_debug.gpkg


In [25]:
from pathlib import Path
import pandas as pd

BASE = Path("/Users/Tomo/Desktop/Research/Data")
OUT_DIR = BASE / "Intermediate" / "Census2020" / "tokyo_stat_debug"
OUT_DIR.mkdir(parents=True, exist_ok=True)

tokyo_stat = merged[
    merged["市区町村コード"].astype(str).str.startswith("13")
].copy()

print("tokyo_stat shape:", tokyo_stat.shape)
print(tokyo_stat.columns.tolist())

out_csv = OUT_DIR / "tokyo_stat_final_key_check.csv"
tokyo_stat.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("saved:", out_csv)

tokyo_stat shape: (5522, 59)
['final_key', '市区町村コード', 'final_unit_code', '総数', '親族のみの世帯', '核家族世帯', 'うち夫婦のみの世帯', 'うち夫婦と子供から成る世帯', '核家族以外の世帯', '非親族を含む世帯', '単独世帯', '世帯の家族類型「不詳」', '（再掲）3世代世帯', '（再掲）夫65歳以上，妻60歳以上の夫婦のみの世帯', '（再掲）65歳以上の単独世帯', '都道府県名', '市区町村名', '大字・町名', '字・丁目名', '地域階層レベル', '秘匿処理', '町丁字コード', '町丁字コード_norm', 'source_table', '住宅に住む一般世帯', '持ち家', '公営・都市再生機構・公社の借家', '民営の借家', '給与住宅', '間借り', '住宅以外に住む一般世帯', '住居の種類「不詳」', 'pop15_total', '労働力人口', '非労働力人口', '労働力状態「不詳」', '0_総数', 'A_管理的職業従事者', 'B_専門的・技術的職業従事者', 'C_事務従事者', 'D_販売従事者', 'E_サービス職業従事者', 'F_保安職業従事者', 'G_農林漁業従事者', 'H_生産工程従事者', 'I_輸送・機械運転従事者', 'J_建設・採掘従事者', 'K_運搬・清掃・包装等従事者', 'L_分類不能の職業', 'old_couple_rate', 'old_single_rate', 'single_parent_proxy_rate', 'rent_rate', 'sales_service_rate', 'agriculture_rate', 'blue_collar_rate', 'unemployment_rate', 'exclude_nonresidential', 'ADI_raw']
saved: /Users/Tomo/Desktop/Research/Data/Intermediate/Census2020/tokyo_stat_debug/tokyo_stat_final_key_check.csv
